In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)

GROQ_BASE_URL = os.getenv('GROQ_BASE_URL')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

groq = OpenAI(base_url = GROQ_BASE_URL, api_key = GROQ_API_KEY)


In [3]:
system_message = """
You are a helpful assistant!!!
"""

In [4]:
def chatbot(history):
    history = [{'role':h['role'], 'content':h['content']} for h in history]
    messages = [{'role':'system', 'content':system_message}] + history
    response = groq.chat.completions.create(
        model = 'openai/gpt-oss-20b',
        messages = messages
    )

    reply = response.choices[0].message.content

    return history + [{'role':'assistant', 'content':reply}]

    # chatbot box expects the entire conversation but not simply the output response


In [5]:
def put_message_in_chatbot(message, history):
    return "", history + [{'role':'user', 'content':message}]

    # after hitting the enter button or submitting, input message box get's replaced with empty string
    # and the conversation box get's populated with the history

#### Custom UI development using Gradio Blocks

In [6]:
with gr.Blocks() as ui:

    with gr.Row():
        conversation = gr.Chatbot(height=250)                       # chatbot box

    with gr.Row():
        input_box = gr.Textbox(label='Chat with the AI Assistant')  # input message box

    input_box.submit(
        put_message_in_chatbot,
        inputs=[input_box, conversation],       # takes input from both input_box and conversation
        outputs=[input_box, conversation]       # returns the input_box with "" (empty string) and conversation with chat history
    ).then(
        chatbot,
        inputs=conversation,                    # takes input from the conversation [chat history]
        outputs=conversation                    # returns the output to conversation with the history and the latest reply
    )

    ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7887
* To create a public link, set `share=True` in `launch()`.
